In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# **Data Cleaning**

First we will load in the scraped data so we can inspect it and identify any entries that need to be cleaned


In [15]:
# Load in the scraped data
df = pd.read_csv("retail_raw.csv")

The following cells give us an overview of the raw data

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1900 entries, 0 to 1899
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product       1900 non-null   object 
 1   sale_date     1900 non-null   object 
 2   category      1900 non-null   object 
 3   quantity      1900 non-null   int64  
 4   total_price   1900 non-null   float64
 5   total_profit  1900 non-null   float64
 6   payment_type  1900 non-null   object 
 7   customer_id   1900 non-null   int64  
 8   location      1900 non-null   object 
 9   gender        1900 non-null   object 
 10  age_band      1889 non-null   object 
 11  year          1900 non-null   int64  
 12  quarter       1900 non-null   int64  
 13  page          1900 non-null   int64  
dtypes: float64(2), int64(5), object(7)
memory usage: 207.9+ KB


In [17]:
df.describe(include="all")

,product,sale_date,category,quantity,total_price,total_profit,payment_type,customer_id,location,gender,age_band,year,quarter,page
count,1900,1900,1900,1900.000000,1900.000000,1900.000000,1900,1900.000000,1900,1900,1889,1900.0,1900.000000,1900.000000
unique,98,356,42,NaN,NaN,NaN,9,NaN,26,5,67,NaN,NaN,NaN
top,Blue Floral Embroidered Mac,10/12/2025,Children — Girls Clothing,NaN,NaN,NaN,Credit Card,NaN,Dublin,Female,Category 25-34,NaN,NaN,NaN
freq,57,12,380,NaN,NaN,NaN,903,NaN,603,719,121,NaN,NaN,NaN
mean,NaN,NaN,NaN,1.567368,52.348368,22.516937,NaN,9610.744211,NaN,NaN,NaN,2025.0,2.515263,8.426316
std,NaN,NaN,NaN,1.121943,52.369199,19.927423,NaN,5374.742135,NaN,NaN,NaN,0.0,1.108766,4.579454
min,NaN,NaN,NaN,-1.000000,6.750000,3.080000,NaN,1000.000000,NaN,NaN,NaN,2025.0,1.000000,1.000000
25%,NaN,NaN,NaN,1.000000,29.000000,11.600000,NaN,5086.250000,NaN,NaN,NaN,2025.0,2.000000,4.000000
50%,NaN,NaN,NaN,1.000000,35.000000,17.590000,NaN,8962.000000,NaN,NaN,NaN,2025.0,3.000000,8.000000
75%,NaN,NaN,NaN,2.000000,58.000000,26.500000,NaN,14114.500000,NaN,NaN,NaN,2025.0,4.000000,12.000000


From the above table we can see:  
9 diff types of `payment_type` but there should only be 4: credit card, debit card, voucher or Paypal    
5 diff `gender` should only be 3: male, female or prefer not to say.   
67 diff `age_bands` should be 7: 18-24, 25-34, 35-44, 45-54, 55-64, 65+ and None   
we'll assume the None values are people not specifying their age rather than data losses.

The `sale_date` column needs to be converted from text object to datetime format so that time-based analysis can be performed

## Payment type cleaning

before cleaning the different payment types are:

In [19]:
df["payment_type"].unique()

array(['Credit Card', 'Debit Card', 'CC', 'PayPal', 'Voucher',
       'Debit  Card', 'Credit  Card', 'Pay  Pal', 'Pay Pal'], dtype=object)

To clean this category we can run the following

In [22]:
df["payment_type"] = (
    df["payment_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .replace({
        "CC": "Credit Card",
        "Credit Card": "Credit Card",
        "Credit  Card": "Credit Card",
        "Debit Card": "Debit Card",
        "Debit  Card": "Debit Card",
        "PayPal": "PayPal",
        "Pay Pal": "PayPal",
        "Pay  Pal": "PayPal",
        "Voucher": "Voucher"
    })
)
df["payment_type"].unique()

array(['Credit Card', 'Debit Card', 'PayPal', 'Voucher'], dtype=object)

## Gender cleaning

before cleaning the different types of gender are:

In [20]:
df["gender"].unique()

array(['Female', 'F', 'Male', 'Prefer not to say', 'M'], dtype=object)

In [ ]:
df["gender"] = (
    df["gender"]
    .astype(str)
    .str.strip()
    .replace({
        "F": "Female",
        "Female": "Female",
        "M": "Male",
        "Male": "Male",
        "Prefer not to say": "Prefer not to say"
    })
)

## Age Band Cleaning

Before cleaning the different Age bands are:

In [18]:
df["age_band"].unique()

array(['Range 25-34', 'Group 35—44', 'Category 55—64', 'Range 35-44',
       'Group 35-44', 'Group 45-54', 'Range 18-24', 'Category 35-44',
       'Category 25-34', '25–34', '18-24', 'Range 55-64', '35–44',
       'Group 25-34', nan, 'Range 35—44', 'Category 18–24',
       'Category 55-64', 'Group 18-24', 'Group 25–34', '55-64',
       'Category 18—24', '25-34', 'Range 25—34', '45—54', 'Group 55–64',
       'Group 65+', 'Category 65+', '45-54', '35-44', 'Category 25–34',
       'Range 25–34', 'Group 18—24', 'Category 35—44', 'Category 35–44',
       'Group 45—54', 'Group 45–54', '55–64', 'Group 18–24', '55—64',
       'Category 45-54', 'Range None', '65+', '18–24', 'Category 25—34',
       'Category 18-24', 'Range 45-54', 'Group 35–44', 'Range 45–54',
       'Range 18—24', '35—44', 'Category 45—54', 'Group 55-64',
       'Range 55—64', '45–54', '18—24', 'Category 55–64', 'Group 25—34',
       '25—34', 'Group 55—64', 'Range 35–44', 'Group None', 'Range 18–24',
       'Range 45—54', 'Cat

# **Data Analysis**